# Opensolr

> [Opensolr](https://opensolr.com) is a managed Apache Solr platform (since 2011) with vector-enabled Solr 9.x environments, server-side GPU embeddings (multilingual E5-large-instruct, 1024-dim), and a native `{!hybrid}` query parser that fuses BM25 and kNN scores per document.

This notebook covers the `OpensolrVectorStore` and `OpensolrEmbeddings` integrations.

What makes it different from most vector stores here:
- **No embedding model to configure** — texts and queries are embedded server-side. The `embedding` argument is not needed.
- **Hybrid search built in** — `hybrid=True` runs lexical BM25 and semantic kNN in one Solr query and fuses scores, with a tunable balance.
- **It is plain Apache Solr underneath** — every index is also reachable over the native Solr API (`/select`, facets, highlighting) for anything beyond retrieval.

## Setup

This notebook runs as-is on a shared demo account — press Enter at both prompts below to use it:

- email: `mcp@opensolr.com`
- API key: `420b8b23e7b12dc8ab838932145a5065`
- preloaded index: `mcp_demo_d1__dense` (300 news articles)

The account is shared publicly, so other people can change or delete what you create, and anything created there is deleted after 3 days, automatically. Limits are per index and small on purpose: 200 MB bandwidth, 50 MB disk.

For a private index that persists, create a free account at [opensolr.com/register](https://opensolr.com/register) (15-day trial, no card) and copy your API key from **Account**.

In [ ]:
%pip install -qU langchain-opensolr

In [ ]:
import getpass

# Press Enter at either prompt to fall back to the shared demo account.
OPENSOLR_EMAIL = input("Opensolr email [mcp@opensolr.com]: ") or "mcp@opensolr.com"
OPENSOLR_API_KEY = (
    getpass.getpass("Opensolr API key [demo key]: ")
    or "420b8b23e7b12dc8ab838932145a5065"
)

## Initialization

`create_if_missing=True` provisions a vector-enabled index automatically. Vector indexes live in Opensolr's Solr 9.x locations — currently `us` (Chicago), `de` (Germany) or `fi` (Finland); the list is fetched live from the platform, and additional dedicated regions can be deployed on request (support@opensolr.com).

In [ ]:
from langchain_opensolr import OpensolrVectorStore

vector_store = OpensolrVectorStore(
    index="langchain_demo__dense",
    email=OPENSOLR_EMAIL,
    api_key=OPENSOLR_API_KEY,
    location="us",
    create_if_missing=True,
)

## Manage vector store

### Add items

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(page_content="Hybrid search fuses BM25 with vector similarity", metadata={"category": "search"}),
    Document(page_content="Cats sleep sixteen hours a day", metadata={"category": "animals"}),
    Document(page_content="The recipe calls for two cups of flour", metadata={"category": "cooking"}),
]

vector_store.add_texts(
    [d.page_content for d in documents],
    metadatas=[d.metadata for d in documents],
    ids=["doc1", "doc2", "doc3"],
)

### Delete items

In [ ]:
vector_store.delete(ids=["doc3"])

## Query vector store

### Query directly

In [ ]:
results = vector_store.similarity_search("how do search engines combine relevance signals?", k=2)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

With scores:

In [ ]:
results = vector_store.similarity_search_with_score("sleepy pets", k=1)
for doc, score in results:
    print(f"* [SIM={score:.3f}] {doc.page_content}")

### Hybrid search

`hybrid=True` runs BM25 and kNN together via Opensolr's `{!hybrid}` parser. `alpha` moves the balance (0 = all semantic, 1 = all lexical); `mode` controls how the two sides combine (`union`, `keywords_required`, `meaning_required`, `intersection`).

In [ ]:
results = vector_store.similarity_search(
    "BM25 relevance", k=2, hybrid=True, mode="union", alpha=0.5
)
for doc in results:
    print(f"* {doc.page_content}")

### Metadata filtering

In [ ]:
vector_store.similarity_search("anything", k=5, filter={"category": "animals"})

### Query by turning into retriever

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2, "hybrid": True})
retriever.invoke("how does hybrid retrieval work?")

## Usage for retrieval-augmented generation

For guides on using this vector store for RAG, see:

- [Tutorials](https://python.langchain.com/docs/tutorials/rag/)
- [How-to: Question and answer with RAG](https://python.langchain.com/docs/how_to/#qa-with-rag)
- [Retrieval conceptual docs](https://python.langchain.com/docs/concepts/retrieval/)

## API reference

- Package: [github.com/opensolr/langchain-opensolr](https://github.com/opensolr/langchain-opensolr)
- Opensolr platform docs: [AI & Vector Search](https://opensolr.com/opensolr-platform-user-documentation/ai-vector)